# Множество Парето для реконструкции треков

Статья Андрея Чернова: облако конфигураций GNN и декодеров в координатах
**качество полных треков — полная задержка обработки**.

Код импортируется из [fixed-target-exp](https://github.com/DNScookie/fixed-target-exp).
Внутри ноутбука нет второй копии генератора, сетей и метрик. Коммит зафиксирован.

Это новый протокол v2. Старые числа курсовой здесь не объявляются воспроизведёнными.
Исправлены генератор, эталон по частицам, сопоставление треков и границы времени;
декодеры унифицированы. Подробности находятся в `experiments/pareto/README.md`.

В Colab выберите GPU и выполните ячейки сверху вниз. Сначала оставьте `smoke`:
это проверка выполнения, её качество не является результатом для статьи.
Обучение в этом экземпляре ноутбука заранее не выполнено.


## 1 Фиксированная версия проекта

Загрузка публичного репозитория не требует токена. Существующий каталог с изменениями
не перезаписывается. Для новой версии следует явно заменить `REPO_COMMIT`.


In [ ]:
import os, sys, subprocess
from pathlib import Path
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
REPO_URL = "https://github.com/DNScookie/fixed-target-exp.git"
REPO_COMMIT = "3020d2470e501908bb13a8142114b7f2557ed1d6"
REPO_DIR = Path("/content") / ("fixed-target-exp-" + REPO_COMMIT[:8])
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    subprocess.check_call(["git", "checkout", "--detach", REPO_COMMIT], cwd=REPO_DIR)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True).strip()
if actual != REPO_COMMIT or dirty:
    raise RuntimeError("Версия проекта отличается или есть локальные изменения; используйте чистый каталог.")
sys.path.insert(0, str(REPO_DIR))
os.environ["FIXED_TARGET_COMMIT"] = actual
print("Repository commit:", actual)


## 2 Зависимости

Используем встроенный PyTorch Colab. Его и CUDA не переустанавливаем.
Для используемых базовых слоёв PyG расширения scatter/sparse необязательны:
https://pytorch-geometric.readthedocs.io/en/stable/notes/installation.html


In [ ]:
import importlib.util
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Нужен Colab со встроенным PyTorch либо локальная установка PyTorch.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO_DIR / "requirements.txt"),
    "-r", str(REPO_DIR / "experiments/pareto/requirements.txt")])
from experiments.pareto.core import *
from experiments.pareto.study import *
from experiments.pareto.plots import *
print("Device:", DEVICE)
print("Code hash:", CODE_HASH)


## 3 Профиль и сохранение

`smoke` — 24 конфигурации и 3 эпохи; `screen` — 252 конфигурации и 30 эпох;
`paper` — 896 конфигураций на каждый из трёх seed и 60 эпох.
Оцените длительность короткого запуска перед большим перебором.

Для длительных запусков включите Google Drive: стандартное окно Colab запросит доступ.
Без Drive результаты в `/content` исчезнут после сброса среды.
Можно загрузить старый `dataset.json` и указать путь: координаты сохранятся,
но новые метрики всё равно будут отличаться. При нехватке событий явно уменьшите
размеры разбиений в `PRESETS[PROFILE]`.


In [ ]:
PROFILE = "smoke"  # затем "screen"; "paper" после проверки основного опыта
USE_GOOGLE_DRIVE = False
LEGACY_DATASET = None  # например "/content/dataset.json"
RUN_TEST = True
RUN_ROBUSTNESS = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/TrackingPareto"
else:
    OUTPUT_ROOT = "/content/TrackingPareto"
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
display(PRESETS[PROFILE])


## 4 Численные проверки до обучения

Проверяем генератор и предел аналитического движения; доминирование, неизменность
эталона после фильтрации, штраф за дубликаты и непересекающиеся пути обоих декодеров.
Эти проверки уже выполнены локально. Следующие ячейки повторяют их на вашей среде.


In [ ]:
subprocess.check_call([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO_DIR)
subprocess.check_call([sys.executable, "-m", "tests.check_pareto_core"], cwd=REPO_DIR)


## 5 Валидационный перебор

Веса определяются на train, лучшая эпоха — по validation BCE. Перебор порогов,
ширины поиска и фильтрации проводится на validation. Эталон состоит из частиц
с минимум тремя исходными хитами; после фильтрации он неизменен. Сопоставление
взаимно однозначно при purity и completeness ≥ 0,8. F1 — micro по событиям,
затем среднее по seed обучения.

У фильтрованного и исходного вариантов одни веса классификатора рёбер. Предсказания
кэшируются только для сканирования качества. Для каждого замера времени весь путь
выполняется заново: CPU-хиты → граф → переносы → фильтр и обрезка → сеть рёбер →
sigmoid → CPU-декодер. Симуляция событий и оценка качества исключены. Есть прогрев
и синхронизация CUDA. Время разных GPU не объединяется в одну серию.

Этот выпуск обучается и оценивается при `k_values=[1]`. Изменение графа обучения —
отдельный эксперимент. Веса сохраняются после завершения обучения каждой сети;
при обрыве текущая сеть обучается заново, завершённые веса используются повторно.


In [ ]:
state = run_validation(PROFILE, OUTPUT_ROOT, LEGACY_DATASET)
print("Results:", state["session"])
display(state["summary"].sort_values("time_ms").head(20))
print("Выбраны по validation:", state["selected"])


## 6 Главный рисунок и сопутствующие графики

Сохраняются всё облако и общий фронт с включённой стоимостью фильтра, качество
при бюджете времени, F1 рёбер против F1 треков и чувствительность к порогу
при фиксированных модели, фильтре и k. Каждая кривая порога имеет фиксированный B.
CSV содержат все точки, включая доминируемые, и устойчивые `config_id`.
Кривые бюджетов ступенчатые: промежуточные конфигурации не выдумываются.


In [ ]:
plot_frontier(state)


## 7 Отложенный тест

По validation уже записан `selected_before_test.json`: до пяти конфигураций
для разных бюджетов. Только они проверяются на test. Новые пороги по тесту
не выбираются, общий тестовый фронт не строится. Если после просмотра теста
менять алгоритм, понадобится новая отложенная выборка.
Bootstrap-интервалы качества описательные, особенно при одном или трёх seed.


In [ ]:
if RUN_TEST:
    display(run_test(state))
    display(bootstrap_test(state))


## 8 Устойчивость без повторного подбора

Необязательные два однофакторных опыта: 25, 50 и 100 частиц; отдельно коэффициент
фона 0, 0,4 и 0,8. Веса и конфигурации фиксированы. Это дешевле полной сетки.
При импорте старого JSON проверка запрещена: новый генератор не воспроизводит
его распределение. Сначала завершите основной опыт.


In [ ]:
if RUN_ROBUSTNESS:
    display(robustness_check(state))


## 9 Кривые обучения и экспорт

Архив сессии содержит CSV, конфигурации, среду, исходные повторы времени и графики.
Полный архив дополнительно содержит данные и веса. Для разбора первого запуска
передайте архив сессии; при длительной работе сохраняйте полный каталог на Drive.
Каждый рисунок должен ссылаться на свою числовую серию.


In [ ]:
loss_files = sorted(state["runroot"].glob("*.loss.csv"))
if loss_files:
    fig, ax = plt.subplots(figsize=(8, 4))
    for path in loss_files:
        loss = pd.read_csv(path)
        ax.plot(loss.epoch, loss.val_loss, label=path.stem)
    ax.set(xlabel="Epoch / Эпоха", ylabel="Validation BCE")
    ax.legend(fontsize=7); ax.grid(alpha=.2)
    save_plot(fig, state["session"] / "07_learning_curves")

import shutil
session_archive = shutil.make_archive(str(state["session"])+"_results", "zip", state["session"])
full_archive = shutil.make_archive(str(state["runroot"])+"_full", "zip", state["runroot"])
print("Сессия:", session_archive)
print("Данные и веса:", full_archive)
try:
    from google.colab import files
    files.download(session_archive)
except ImportError:
    pass
